In [ ]:
# Set Kaggle API credentials as environment variables so the Kaggle CLI can authenticate
import os
os.environ['KAGGLE_USERNAME']= 'surieyaa'
os.environ['KAGGLE_KEY']='KGAT_f4e51cbb08019bceb3f3029b435ca847'


In [ ]:
# Download the Cardiovascular Diseases Risk Prediction dataset from Kaggle
!kaggle datasets download -d 'alphiree/cardiovascular-diseases-risk-prediction-dataset'


In [ ]:
# Load the downloaded dataset (zipped CSV) into a pandas DataFrame
import pandas as pd
df = pd.read_csv('/content/cardiovascular-diseases-risk-prediction-dataset.zip')


In [ ]:
# Preview the first 10 rows to inspect column names and sample values
df.head(10)


In [ ]:
# Check for missing values in each column
df.isnull().sum()


In [ ]:
# Count how many duplicate rows exist in the dataset
df.duplicated().sum()


In [ ]:
# Check the current shape (rows, columns) of the dataset
df.shape


In [ ]:
# Remove duplicate rows in place and confirm the new shape
df.drop_duplicates(inplace=True)
print(f"Shape of DataFrame after removing duplicates: {df.shape}")


In [ ]:
# Inspect the unique age-range categories present in the data
df['Age_Category'].unique()


In [ ]:
# Print column dtypes and non-null counts for a full schema overview
df.info()


In [ ]:
# Drop the 'Checkup' column (not used as a predictive feature) and preview the result
df.drop('Checkup', axis=1, inplace=True)
display(df.head())


In [ ]:
# Inspect the unique categories in General_Health before encoding it
df['General_Health'].unique()


In [ ]:
# Encode categorical columns into numeric form for modeling
from sklearn.preprocessing import LabelEncoder

# Work on a copy so the original cleaned DataFrame stays untouched
df_enc = df.copy()

# Binary Yes/No columns -> label-encode to 0/1
binary_cols = ['Exercise', 'Skin_Cancer', 'Other_Cancer', 'Depression',
               'Arthritis', 'Sex', 'Smoking_History', 'Heart_Disease']
for col in binary_cols:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col])

# General_Health is ordinal (Poor -> Excellent) -> map explicitly to preserve order,
# rather than relying on arbitrary alphabetical label encoding
health_map = {'Poor':0, 'Fair':1, 'Good':2, 'Very Good':3, 'Excellent':4}
df_enc['General_Health'] = df_enc['General_Health'].map(health_map)

# Age_Category is also ordinal (e.g. '18-24', '25-29', ...) -> sort by the
# numeric lower bound of each range and map to sequential integers
age_map = {cat: i for i, cat in enumerate(sorted(
    df_enc['Age_Category'].unique(),
    key=lambda x: int(x.split('-')[0]) if '-' in x else int(x.replace('+',''))
))}
df_enc['Age_Category'] = df_enc['Age_Category'].map(age_map)

# Diabetes has 4 raw text categories -> collapse into 3 meaningful ordinal levels
# (borderline/pregnancy-only treated as closer to "No" than to a full diagnosis)
diabetes_map = {'No':0, 'No, pre-diabetes or borderline diabetes':1,
                 'Yes, but female told only during pregnancy':1, 'Yes':2}
df_enc['Diabetes'] = df_enc['Diabetes'].map(diabetes_map)

# Preview the fully encoded DataFrame
df_enc.head()


In [ ]:
# Separate features (X) and target (y) for model training
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_enc.drop(columns=['Heart_Disease'])
y = df_enc['Heart_Disease']


In [ ]:
# Split into train/test sets (80/20), stratified so both sets keep the same
# class balance as the full (imbalanced) target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Standardize features: fit the scaler on training data only, then apply
# the same transformation to test data (prevents data leakage)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)


In [ ]:
# Train a Random Forest to later rank feature importance for feature selection
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# class_weight='balanced' compensates for the ~91:9 class imbalance in Heart_Disease
rf = RandomForestClassifier(
    n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)


In [ ]:
# Rank features by importance and keep only the top 10 for the final model
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances)

top_k = 10
selected_features = importances.head(top_k).index.tolist()
print("\nSelected features:", selected_features)


In [ ]:
# Quick sanity check: Random Forest's own accuracy on the test set
# (note: accuracy alone is misleading here due to class imbalance — see
# classification_report / recall checks later for the metrics that matter)
from sklearn.metrics import accuracy_score

y_pred = rf.predict(X_test)
accuracy_score(y_test, y_pred)


In [ ]:
# Subset the scaled train/test sets down to only the RF-selected top features,
# since the SVM will be trained on this reduced, higher-signal feature set
X_train_sel = X_train_scaled[selected_features]
X_test_sel = X_test_scaled[selected_features]


In [ ]:
# --- Attempt 1: hyperparameter search using HalvingRandomSearchCV ---
# NOTE: this approach was later found to be unreliable on this imbalanced target —
# its aggressive resource-shrinking produces tiny CV folds that can end up with
# zero positive examples, causing meaningless near-random scores. Kept here for
# reference; the corrected approach is in the next cell.
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV

# Stratified tuning subsample — full SVC training is O(n^2)-O(n^3), impractical on 247K rows
X_tune, _, y_tune, _ = train_test_split(
    X_train_sel, y_train, train_size=8000, stratify=y_train, random_state=42
)

# Hyperparameter grid to search over
param_dist = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 0.01, 0.1, 1],
    'kernel': ['rbf']
}

# max_iter caps runaway fits; cache_size speeds up RBF kernel computation
svm_search = HalvingRandomSearchCV(
    SVC(class_weight='balanced', random_state=42, cache_size=1000, max_iter=5000),
    param_distributions=param_dist,
    factor=3,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=2
)
svm_search.fit(X_tune, y_tune)

print("Best params:", svm_search.best_params_)
print("Best CV ROC-AUC:", svm_search.best_score_)


In [ ]:
# --- Attempt 2 (corrected): hyperparameter search using plain RandomizedSearchCV ---
# Fixes the fold-imbalance problem from HalvingRandomSearchCV by evaluating every
# candidate on the FULL 8000-row tuning set each time (no resource shrinking),
# so every CV fold reliably contains enough positive examples for stable scoring
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV, train_test_split

# Stratified tuning subsample — 8000 keeps ~720 positives, enough for stable folds
X_tune, _, y_tune, _ = train_test_split(
    X_train_sel, y_train, train_size=8000, stratify=y_train, random_state=42
)

# Hyperparameter grid to search over (dropped 'poly' kernel — main cause of slow fits)
param_dist = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 0.01, 0.1, 1],
    'kernel': ['rbf']
}

# n_iter=8 candidates x cv=3 folds = 24 total fits; max_iter/cache_size keep it fast
svm_search = RandomizedSearchCV(
    SVC(class_weight='balanced', random_state=42, cache_size=1000, max_iter=10000),
    param_distributions=param_dist,
    n_iter=8,
    scoring='roc_auc',
    cv=3,             # standard 3-fold, no resource shrinking — folds stay full-size
    n_jobs=-1,
    random_state=42,
    verbose=2
)
svm_search.fit(X_tune, y_tune)

print("Best params:", svm_search.best_params_)
print("Best CV ROC-AUC:", svm_search.best_score_)


In [ ]:
# Fit the final SVM using the best hyperparameters found above
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

# Train on a larger 40,000-row stratified subsample (full 247K rows is still
# impractical for SVC's O(n^2)-O(n^3) training complexity)
X_final, _, y_final, _ = train_test_split(
    X_train_sel, y_train, train_size=40000, stratify=y_train, random_state=42
)

# class_weight='balanced' compensates for class imbalance;
# probability=True enables predict_proba() for later threshold tuning
best_svm = SVC(
    **svm_search.best_params_,
    class_weight='balanced',
    probability=True,
    cache_size=1000,
    max_iter=10000,
    random_state=42
)
best_svm.fit(X_final, y_final)


In [ ]:
# Evaluate the SVM on the held-out test set at the DEFAULT 0.5 threshold
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve
import matplotlib.pyplot as plt

y_pred = best_svm.predict(X_test_sel)
y_proba = best_svm.predict_proba(X_test_sel)[:, 1]

# Precision/recall/F1 per class, plus overall ROC-AUC
print(classification_report(y_test, y_pred, target_names=['No', 'Yes']))
print("Test ROC-AUC:", roc_auc_score(y_test, y_proba))

# Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['No', 'Yes']).plot(cmap='Blues')
plt.title('Confusion Matrix - Optimized SVM')
plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_proba):.3f}")
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Optimized SVM'); plt.legend()
plt.show()


In [ ]:
# Find the classification threshold that maximizes F1-score on the minority
# ("Yes") class, since the default 0.5 cutoff performs poorly under class imbalance
from sklearn.metrics import precision_recall_curve
import numpy as np

precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_idx = np.argmax(f1_scores[:-1])  # last point has no corresponding threshold
best_threshold = thresholds[best_idx]

print(f"Best threshold: {best_threshold:.3f}")
print(f"Precision at best threshold: {precisions[best_idx]:.3f}")
print(f"Recall at best threshold: {recalls[best_idx]:.3f}")
print(f"F1 at best threshold: {f1_scores[best_idx]:.3f}")


In [ ]:
# Re-evaluate the model using the TUNED threshold instead of the default 0.5
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_pred_tuned = (y_proba >= best_threshold).astype(int)
print(classification_report(y_test, y_pred_tuned, target_names=['No', 'Yes']))

# Confusion matrix at the new, tuned threshold
cm = confusion_matrix(y_test, y_pred_tuned)
ConfusionMatrixDisplay(cm, display_labels=['No', 'Yes']).plot(cmap='Blues')
plt.title(f'Confusion Matrix - Threshold={best_threshold:.2f}')
plt.show()


In [ ]:
# Visualize precision/recall tradeoff across thresholds
plt.figure(figsize=(8,5))
plt.plot(thresholds, precisions[:-1], label='Precision')
plt.plot(thresholds, recalls[:-1], label='Recall')
plt.plot(thresholds, f1_scores[:-1], label='F1', linestyle='--')
plt.axvline(best_threshold, color='gray', linestyle=':', label=f'Best threshold={best_threshold:.2f}')
plt.xlabel('Threshold'); plt.ylabel('Score')
plt.title('Precision-Recall-F1 vs Threshold')
plt.legend()
plt.show()


In [ ]:
# Save the trained model and all inference-time dependencies as a single pickle file
import pickle

# Bundle everything needed for inference into one dict
artifact = {
    'model': best_svm,
    'scaler': scaler,
    'selected_features': selected_features,
    'health_map': health_map,
    'age_map': age_map,
    'diabetes_map': diabetes_map,
    'binary_cols': binary_cols,
    'best_threshold': best_threshold
}

with open('cvd_svm_model.pkl', 'wb') as f:
    pickle.dump(artifact, f)

print("Saved cvd_svm_model.pkl")
